# VNetra-Lite — Analisis Kuantitatif Data Sesi Pengujian

**Program Studi Teknik Elektro | Skripsi S1**

---

## Tentang Notebook

Notebook ini adalah instrumen analisis kuantitatif untuk memvalidasi kinerja sistem *Electronic Travel Aid* (ETA) VNetra-Lite.
Data diperoleh dari `SessionDataLogger.kt` yang merekam setiap siklus `evaluateObstacles()` selama sesi pengujian lapangan.

| Figur | Judul | Tujuan |
|:---:|:---|:---|
| **Fig. 4.1** | Threshold Adaptif vs Waktu | Validasi respon dinamis |
| **Fig. 4.2** | Kecepatan vs Threshold | Validasi formula Tanh vs data empiris |
| **Fig. 4.3** | Dekomposisi Latensi | Analisis bottleneck komunikasi |
| **Fig. 4.4** | Packet Loss dan PDR | Reliabilitas transmisi UDP |

**Cara Penggunaan:**
1. Salin `VNetra_Session_*.csv` dari Android ke folder `analysis/` ini.
2. Sesuaikan nilai di Sel Konfigurasi jika diperlukan.
3. Untuk data nyata: *comment* sel Dummy, *uncomment* sel Load CSV.
4. Jalankan semua sel secara berurutan. Gambar disimpan di `output/` (300 DPI).

In [ ]:
# !pip install pandas matplotlib numpy
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# ── Gaya plot: Q1 Journal Standard (IEEE / Elsevier) ──────────────────────
# Mengacu pada IEEE Author Guidelines & Elsevier Artwork Instructions:
# - Latar putih, teks hitam, sans-serif (Arial equiv.), minimal ink.
# - Grid tipis abu-abu, tanpa frame tebal, legend bersih.
# - Resolusi cetak 300 DPI, ukuran font 9-10 pt.
plt.rcParams.update({
    # Canvas
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
    'axes.edgecolor':     '#333333',
    'axes.linewidth':     0.8,
    # Text
    'text.color':         '#111111',
    'axes.labelcolor':    '#111111',
    'xtick.color':        '#333333',
    'ytick.color':        '#333333',
    'axes.labelsize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'axes.titlesize':     10,
    'axes.titleweight':   'bold',
    # Font — sans-serif mirip Arial (IEEE standard)
    'font.family':        'sans-serif',
    'font.sans-serif':    ['DejaVu Sans', 'Arial', 'Helvetica'],
    # Grid — tipis, tidak mengganggu data
    'axes.grid':          True,
    'grid.color':         '#CCCCCC',
    'grid.linewidth':     0.5,
    'grid.linestyle':     '--',
    'grid.alpha':         0.7,
    # Tick
    'xtick.direction':    'in',
    'ytick.direction':    'in',
    'xtick.major.size':   3.5,
    'ytick.major.size':   3.5,
    # Legend
    'legend.facecolor':   'white',
    'legend.edgecolor':   '#AAAAAA',
    'legend.fontsize':    9,
    'legend.framealpha':  0.9,
    # Savefig
    'figure.dpi':         100,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.facecolor':  'white',
})
os.makedirs('output', exist_ok=True)

# ── Palet warna Q1 Journal (color-blind safe, WCAG-compliant) ───────────────
# Menggunakan subset dari Wong (2011) color-blind safe palette:
# Nature Methods 8(6):441 — direkomendasikan untuk publikasi ilmiah.
C_THRESHOLD = '#0072B2'   # Biru tua    — Threshold T (IBM Blue)
C_DISTANCE  = '#D55E00'   # Oranye tua  — Jarak objek (vermillion)
C_THEORY    = '#E69F00'   # Oranye muda — Kurva teoritis (orange)
C_ALERT     = '#CC79A7'   # Ungu muda   — Momen alert
C_SAFE      = '#009E73'   # Hijau tua   — d0 (bluish green)
C_MAX       = '#D62728'   # Merah       — T_max
C_SCATTER   = '#0072B2'   # Biru        — Scatter data lapangan
print('Lingkungan analisis Q1-journal siap.')

---
## Sel 2 — Konfigurasi Parameter Formula

Semua konstanta di bawah ini **harus konsisten** dengan `VNetraConfig.kt` dan nilai yang didokumentasikan di **Bab 3**.

| Parameter | Simbol | Nilai | Sumber Literatur |
|:---|:---:|:---:|:---|
| Jarak aman minimum | $d_0$ | 1000 mm | Desain sistem (1 m konservatif) |
| Batas threshold maksimum | $T_{max}$ | 4000 mm | Jangkauan valid VL53L5CX |
| Perception-Reaction Time | $t_R$ | 1.3 s | Kovacs & Nagy (2020) [44] |
| Durasi langkah penuh | $t_{step}$ | 0.63 s | Knoblauch et al. [42]; Winter [45] |

In [ ]:
# Sesuaikan dengan nilai di VNetraConfig.kt Anda
D0     = 1000    # BASE_WARNING_DIST_MM (mm)
T_MAX  = 4000    # MAX_THRESHOLD_MM (mm)
T_R    = 1.3     # PERCEPTION_REACTION_TIME_SEC — Kovacs & Nagy [44]
T_STEP = 0.63    # STEP_DURATION_SEC — Knoblauch et al. [42]; Winter [45]
print(f'Config: d0={D0}mm | T_max={T_MAX}mm | t_R={T_R}s | t_step={T_STEP}s')

---
## Sel 3 — Generator Data Dummy

> **Catatan:** Blok ini menghasilkan data **simulasi** untuk pengembangan. Setelah pengujian lapangan, ganti dengan sel Load CSV di bawahnya.

Generator menggunakan **siklus periodik** (~120 s/siklus) sehingga realistis untuk durasi berapa pun (60 s, 600 s, 3600 s).

Setiap siklus terdiri dari empat fase:
1. **Idle** (10 s): Sistem diam, halangan jauh (~4500 mm)
2. **Approach** (60 s): Berjalan mendekati halangan, kecepatan ~1.0–1.3 m/s
3. **Alert** (30 s): Halangan dalam jangkauan, sistem melambat dan memperingatkan
4. **Recover** (20 s): Pengguna menghindari, halangan kembali menjauh

In [ ]:
def generate_dummy(fname='VNetra_Session_DUMMY.csv', duration_s=60, fps=15):
    n  = int(duration_s * fps)
    t  = np.linspace(0, duration_s, n)
    np.random.seed(42)
    dt = 1.0 / fps

    # ── Profil kecepatan + jarak objek berbasis siklus ─────────────────────
    # Setiap siklus: idle(10s) → approach(60s) → alert(30s) → recover(20s)
    CYCLE  = 120.0   # durasi satu siklus (s)
    T_IDLE = 10.0; T_APP = 60.0; T_ALT = 30.0  # fase dalam siklus

    v_raw  = np.zeros(n)
    d_obj  = np.full(n, 4500.0)

    for i, ti in enumerate(t):
        phase = ti % CYCLE
        if phase < T_IDLE:
            # Diam: tidak ada kecepatan, halangan jauh
            v_raw[i] = 0.0
            d_obj[i] = 4500.0
        elif phase < T_IDLE + T_APP:
            # Mendekati halangan: kecepatan naik bertahap lalu stabil
            p = (phase - T_IDLE) / T_APP   # 0–1
            v_raw[i] = 1200.0 * min(p * 4, 1.0)   # ramp up 0→1200 mm/s
            d_obj[i] = 4500.0 - 3200.0 * p         # 4500→1300 mm
        elif phase < T_IDLE + T_APP + T_ALT:
            # Alert: melambat, halangan tetap dekat
            p = (phase - T_IDLE - T_APP) / T_ALT
            v_raw[i] = 1200.0 * (1.0 - p)          # ramp down 1200→0
            d_obj[i] = 1300.0 - 500.0 * p           # 1300→800 mm
        else:
            # Recover: berhenti, halangan menjauh kembali
            p = (phase - T_IDLE - T_APP - T_ALT) / (CYCLE - T_IDLE - T_APP - T_ALT)
            v_raw[i] = 0.0
            d_obj[i] = 800.0 + 3700.0 * p           # 800→4500 mm

    # Tambahkan noise sensor
    v_raw = np.maximum(v_raw + np.random.normal(0, 60, n), 0)
    d_obj = np.clip(d_obj + np.random.normal(0, 35, n), 200, 5500).astype(int)
    v_avg = pd.Series(v_raw).ewm(alpha=0.35).mean().values

    # ── Formula threshold ────────────────────────────────────────────────────
    a_lin = np.clip(0.6 + np.random.normal(0, 0.10, n), 0.1, 1.5)
    m_buf = 0.5 * a_lin * 1000 * (T_STEP ** 2)
    ssd   = v_avg * T_R + m_buf
    rv    = float(T_MAX - D0)
    thresh = np.clip(D0 + rv * np.tanh(ssd / rv), D0, T_MAX).astype(int)
    alert  = (d_obj < thresh).astype(int)

    # ── Latensi ────────────────────────────────────────────────────────────
    lhw  = np.random.randint(8,  28, n)
    lnet = np.random.randint(12, 75, n)
    lal  = np.random.randint(2,  7,  n)
    ltts = np.where(alert, np.random.randint(90, 230, n), 0)
    lbt  = np.random.randint(1,  5,  n)

    df = pd.DataFrame({
        'timestamp_ms':    (t * 1000).astype(int) + 1700000000000,
        'elapsed_s':       np.round(t, 2),
        'd_obj_mm':        d_obj,
        'v_raw_mmps':      np.round(v_raw, 2),
        'v_avg_mmps':      np.round(v_avg, 2),
        'm_buffer_mm':     np.round(m_buf, 2),
        'threshold_T_mm':  thresh,
        'alert_triggered': alert,
        'alert_text':      np.where(alert, 'hati-hati halangan', ''),
        'latency_hw_ms':   lhw,  'latency_net_ms':   lnet,
        'latency_algo_ms': lal,  'latency_tts_ms':   ltts,
        'latency_bt_ms':   lbt,  'latency_total_ms': lhw+lnet+lal+ltts+lbt,
        'packet_loss_count': np.random.poisson(0.20, n),
    })
    df.to_csv(fname, index=False)
    n_cycles = duration_s / 120.0
    print(f'Dummy CSV: {fname} | {n} frame | {n_cycles:.1f} siklus | Alert: {int(alert.sum())} frame')
    return df

# Ubah duration_s sesuai kebutuhan (60, 600, 3600)
df = generate_dummy(duration_s=600)
df.head(3)

---
## Sel 4 — Load CSV dari Pengujian Lapangan

Setelah pengujian selesai:
1. *Comment* baris `df = generate_dummy()` di sel di atas
2. *Uncomment* sel ini dan jalankan

In [ ]:
# files = sorted(glob.glob('VNetra_Session_*.csv'))
# if not files:
#     raise FileNotFoundError('Tidak ada CSV! Salin dari Android ke folder ini.')
# print('File ditemukan:', files)
# df = pd.read_csv(files[-1])
# print(f'Loaded: {files[-1]} | {len(df)} baris')
# df.head()

---
## Gambar 4.1 — Perilaku Threshold Adaptif dan Jarak Objek terhadap Waktu

### Tujuan Figur
Figur ini memvalidasi **respons dinamis** sistem. Berbeda dari *static threshold* konvensional (nilai tetap), VNetra-Lite mengadaptasi batas $T$ setiap frame. Figur ini menjawab pertanyaan:

> *"Apakah threshold $T$ benar-benar berubah mengikuti kecepatan gerak, dan apakah selalu menjaga $d_0$ saat sistem diam?"*

### Panduan Membaca Figur
| Elemen Visual | Makna |
|:---|:---|
| Garis biru tebal | Threshold adaptif $T(t)$ — batas peringatan sistem setiap frame |
| Garis oranye | Jarak objek terdekat $d_{obj}(t)$ dari sensor VL53L5CX |
| Simbol segitiga (▲) | Momen alert terpicu — saat $d_{obj} < T$ |
| Garis hijau putus-putus | $d_0 = 1000$ mm — batas aman absolut minimum |
| Garis merah putus-putus | $T_{max} = 4000$ mm — batas jangkauan fisik sensor |

### Interpretasi Hasil
Sistem dikatakan **berhasil** apabila:
1. $T$ naik saat berjalan dan turun saat melambat (adaptif terhadap kecepatan).
2. $T \geq d_0$ sepanjang waktu — tidak pernah turun di bawah zona aman.
3. Alert terpicu *sebelum* $d_{obj}$ menyentuh $d_0$, memberikan jarak reaksi yang cukup.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.5))

ax.plot(df['elapsed_s'], df['threshold_T_mm'],
        color=C_THRESHOLD, lw=1.8, label='Threshold $T$')
ax.plot(df['elapsed_s'], df['d_obj_mm'],
        color=C_DISTANCE, lw=1.4, alpha=0.85, label='Jarak Objek $d_{obj}$')
ax.axhline(D0,    color=C_SAFE, lw=1.2, ls='--', label='$d_0$')
ax.axhline(T_MAX, color=C_MAX,  lw=1.2, ls=':',  label='$T_{max}$')

alerts = df[df['alert_triggered'] == 1]
if len(alerts) > 0:
    ax.scatter(alerts['elapsed_s'], alerts['d_obj_mm'],
               color=C_ALERT, s=20, zorder=5, marker='^',
               edgecolors='none', label='Alert')

ax.set_xlabel('Waktu (s)')
ax.set_ylabel('Jarak (mm)')
ax.set_title('Threshold Adaptif vs Waktu')
ax.set_xlim(df['elapsed_s'].min(), df['elapsed_s'].max())
ax.set_ylim(0, T_MAX * 1.08)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax.legend(loc='upper right', ncol=3, fontsize=8.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('output/Fig41_threshold_vs_waktu.png')
plt.show()
print('Saved: output/Fig41_threshold_vs_waktu.png')

---
## Gambar 4.2 — Kurva Kecepatan vs Threshold (Validasi Saturasi *Tanh*)

### Tujuan Figur
Figur ini adalah **instrumen validasi utama formula**. Setiap titik biru = satu frame pengukuran nyata. Kurva kuning adalah prediksi teoritis dari:

$$T = d_0 + (T_{max} - d_0) \cdot \tanh\!\left(\frac{v_{avg} \cdot t_R + M_{buffer}}{T_{max} - d_0}\right)$$

di mana $M_{buffer} = \frac{1}{2}\,a_{lin}\cdot t_{step}^2$ (dirata-ratakan dari data nyata).

### Dua Properti Kritis
1. **Saat diam** ($v_{avg} \to 0$): $T \to d_0$ — tidak ada ekspansi threshold saat berdiri diam.
2. **Saat berlari** ($v_{avg} \to \infty$): $T \to T_{max}$ — **saturasi**, tidak pernah meledak tanpa batas.

### Interpretasi
Nilai $R^2$ mendekati 1.0 membuktikan implementasi `NavigationCoordinator.kt` sesuai derivasi teori BAB 3. Jika $R^2 < 0.90$, periksa noise IMU atau drift EWMA.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 3.5))

ax.scatter(df['v_avg_mmps'], df['threshold_T_mm'],
           alpha=0.20, s=5, color=C_SCATTER, edgecolors='none', label='Data per frame')

v_line   = np.linspace(0, max(df['v_avg_mmps'].max() * 1.08, 1600), 600)
m_avg    = df['m_buffer_mm'].mean()
ssd_line = v_line * T_R + m_avg
rv       = float(T_MAX - D0)
t_theory = D0 + rv * np.tanh(ssd_line / rv)
ax.plot(v_line, t_theory, color=C_THEORY, lw=2.0, label='Tanh (teoritis)')

ax.axhline(D0,    color=C_SAFE, lw=1.0, ls='--', label='$d_0$')
ax.axhline(T_MAX, color=C_MAX,  lw=1.0, ls=':',  label='$T_{max}$')

t_pred = D0 + rv * np.tanh((df['v_avg_mmps']*T_R + df['m_buffer_mm']) / rv)
ss_res = np.sum((df['threshold_T_mm'] - t_pred)**2)
ss_tot = np.sum((df['threshold_T_mm'] - df['threshold_T_mm'].mean())**2)
r2     = 1 - ss_res/ss_tot if ss_tot > 0 else float('nan')
ax.text(0.97, 0.05, f'$R^2 = {r2:.4f}$',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#AAAAAA'))

ax.set_xlabel('$v_{avg}$ (mm/s)')
ax.set_ylabel('$T$ (mm)')
ax.set_title('Kecepatan vs Threshold (Validasi Tanh)')
ax.set_ylim(D0*0.9, T_MAX*1.04)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('output/Fig42_velocity_vs_threshold.png')
plt.show()
print(f'Saved: output/Fig42_velocity_vs_threshold.png | R2={r2:.4f}')

---
## Gambar 4.3 — Dekomposisi Latensi *End-to-End*

### Tujuan Figur
Sistem ETA efektif mensyaratkan latensi total $< t_R = 1300$ ms (batas reaksi kognitif manusia [44]). Figur ini memperlihatkan kontribusi setiap komponen.

### Komponen Latensi
| Komponen | Kolom CSV | Penjelasan |
|:---|:---|:---|
| **Hardware** | `latency_hw_ms` | Waktu sensor VL53L5CX + pemrosesan ESP32 |
| **Jaringan IoT** | `latency_net_ms` | RTT/2 paket UDP WiFi ESP32 → Android |
| **Algoritma** | `latency_algo_ms` | EWMA + Mahony AHRS + komputasi Tanh |
| **TTS Audio** | `latency_tts_ms` | Render + buffer Android TextToSpeech |
| **Bluetooth** | `latency_bt_ms` | Transmisi audio ke headset BT |

### Panduan Membaca Figur
- **Panel kiri (stacked bar):** Distribusi latensi per segmen waktu. Garis merah = batas $t_R = 1300$ ms.
- **Panel kanan (box plot):** Sebaran statistik per komponen — identifikasi komponen paling tidak stabil.
- Jika batang secara konsisten melebihi garis merah, sistem perlu dioptimasi (kompresi payload UDP, pengurangan teks TTS).

In [ ]:
fig, (ax_bar, ax_box) = plt.subplots(1, 2, figsize=(7.2, 3.5),
                                      gridspec_kw={'width_ratios': [2, 1]})

df2 = df.copy()
df2['time_bin'] = pd.cut(df2['elapsed_s'], bins=15)
grp = df2.groupby('time_bin', observed=True)[[
    'latency_hw_ms','latency_net_ms','latency_algo_ms','latency_tts_ms','latency_bt_ms'
]].mean()
x_ticks  = range(len(grp))
x_labels = [f'{i.left:.0f}' for i in grp.index]
lat_cols = ['latency_hw_ms','latency_net_ms','latency_algo_ms','latency_tts_ms','latency_bt_ms']
lat_lbl  = ['HW','Jaringan','Algo','TTS','BT']
lat_clr  = ['#0072B2','#009E73','#E69F00','#D55E00','#CC79A7']
bottom   = np.zeros(len(grp))
for col, lbl, clr in zip(lat_cols, lat_lbl, lat_clr):
    v = grp[col].fillna(0).values
    ax_bar.bar(x_ticks, v, bottom=bottom, label=lbl, color=clr, alpha=0.85, width=0.72)
    bottom += v
ax_bar.axhline(T_R*1000, color=C_MAX, lw=1.2, ls='--', label=f'$t_R$')
ax_bar.set_xticks(x_ticks)
ax_bar.set_xticklabels(x_labels, rotation=45, fontsize=7)
ax_bar.set_xlabel('Waktu (s)')
ax_bar.set_ylabel('Latensi (ms)')
ax_bar.set_title('(a) Stacked per Segmen')
ax_bar.legend(fontsize=7.5, loc='upper left', ncol=2)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

box_data = [df[c].values for c in lat_cols]
bp = ax_box.boxplot(box_data, patch_artist=True,
                    medianprops=dict(color='black', lw=1.5),
                    whiskerprops=dict(color='#555555', lw=0.8),
                    capprops=dict(color='#555555', lw=0.8),
                    flierprops=dict(marker='.', color='#888888', alpha=0.4, ms=2))
for patch, clr in zip(bp['boxes'], lat_clr):
    patch.set_facecolor(clr); patch.set_alpha(0.6)
ax_box.set_xticks(range(1, 6))
ax_box.set_xticklabels(lat_lbl, fontsize=8)
ax_box.set_ylabel('Latensi (ms)')
ax_box.set_title('(b) Distribusi')
ax_box.axhline(T_R*1000, color=C_MAX, lw=1.0, ls='--')
ax_box.spines['top'].set_visible(False)
ax_box.spines['right'].set_visible(False)

ax_bar.set_ylim(0, 600)
ax_box.set_ylim(0, 600)
fig.suptitle('Dekomposisi Latensi End-to-End', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('output/Fig43_latency_decomposition.png')
lat_mean = df['latency_total_ms'].mean()
plt.show()
print(f'Saved: output/Fig43_latency_decomposition.png | Mean latency: {lat_mean:.1f} ms')

---
## Gambar 4.4 — Reliabilitas Jaringan: Packet Loss dan PDR

### Tujuan Figur
VNetra-Lite menggunakan **UDP** (*connectionless*) — setiap paket yang hilang = satu frame jarak yang tidak diproses Android. Figur ini mengukur:
1. **Pola temporal packet loss** — acak (noise) atau terkonsentrasi (interferensi lokal)?
2. **Packet Delivery Ratio (PDR)** — rasio paket berhasil diterima. Standar minimum: **PDR ≥ 95%** [53].

### Interpretasi
| Pola Kurva | Diagnosis | Tindakan |
|:---|:---|:---|
| Naik landai merata | Noise background normal | Acceptable |
| Loncat tiba-tiba | Interferensi / roaming WiFi | Optimasi kanal |
| PDR ≥ 95% | Memenuhi standar | Laporkan sebagai valid |
| PDR < 95% | Di bawah standar | Pertimbangkan redundansi paket |

In [ ]:
fig, (ax_cum, ax_gauge) = plt.subplots(1, 2, figsize=(7.2, 3.0),
                                        gridspec_kw={'width_ratios': [3, 1]})

cum_loss = df['packet_loss_count'].cumsum()
ax_cum.fill_between(df['elapsed_s'], cum_loss, alpha=0.12, color=C_MAX)
ax_cum.plot(df['elapsed_s'], cum_loss, color=C_MAX, lw=1.5, label='Packet Loss Kumulatif')
ax_cum.set_xlabel('Waktu (s)')
ax_cum.set_ylabel('Packet Loss Kumulatif')
ax_cum.set_title('(a) Packet Loss vs Waktu')
ax_cum.set_xlim(df['elapsed_s'].min(), df['elapsed_s'].max())
ax_cum.legend(fontsize=9)
ax_cum.spines['top'].set_visible(False)
ax_cum.spines['right'].set_visible(False)

total_loss   = int(df['packet_loss_count'].sum())
total_frames = len(df)
pdr = (1 - total_loss / (total_frames + total_loss)) * 100
bar_color = C_SAFE if pdr >= 95 else (C_THEORY if pdr >= 90 else C_MAX)

ax_gauge.barh(['PDR'], [100], color='#EEEEEE', height=0.5, edgecolor='#BBBBBB', lw=0.5)
ax_gauge.barh(['PDR'], [pdr], color=bar_color, height=0.5)
ax_gauge.axvline(95, color='#333333', lw=1.0, ls='--')
ax_gauge.text(pdr/2, 0, f'{pdr:.1f}%', color='white', fontsize=11,
              fontweight='bold', ha='center', va='center')
ax_gauge.set_xlim(0, 100)
ax_gauge.set_xlabel('PDR (%)')
ax_gauge.set_title('(b) PDR')
ax_gauge.tick_params(axis='y', left=False, labelleft=False)
ax_gauge.spines['top'].set_visible(False)
ax_gauge.spines['right'].set_visible(False)
ax_gauge.spines['left'].set_visible(False)

fig.suptitle('Packet Loss dan Packet Delivery Ratio (PDR)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('output/Fig44_packet_loss_pdr.png')
plt.show()
status = 'OK' if pdr >= 95 else 'PERLU INVESTIGASI'
print(f'Saved: output/Fig44_packet_loss_pdr.png | PDR = {pdr:.2f}% [{status}]')

---
## Tabel 4.x — Ringkasan Statistik Pengujian

Tabel ini merangkum metrik kuantitatif untuk **Bab 4 (Hasil dan Pembahasan)**.

> **Panduan pelaporan:** Cantumkan Mean ± SD untuk latensi (bukan hanya rata-rata), dan bandingkan PDR dengan standar literatur [53].

In [ ]:
total_loss = int(df['packet_loss_count'].sum())
pdr      = (1 - total_loss / (len(df) + total_loss)) * 100
lat_mean = df['latency_total_ms'].mean()
lat_std  = df['latency_total_ms'].std()
lat_med  = df['latency_total_ms'].median()
lat_max  = df['latency_total_ms'].max()
fps_avg  = len(df) / df['elapsed_s'].max()
status_lat = 'OK -- Memenuhi' if lat_mean < T_R*1000 else 'PERLU OPTIMASI'
status_pdr = 'OK' if pdr >= 95 else 'PERLU INVESTIGASI'

print('=' * 62)
print('   TABEL RINGKASAN STATISTIK SESI PENGUJIAN VNetra-Lite')
print('=' * 62)
print(f'  Durasi Sesi                  : {df["elapsed_s"].max():.1f} detik')
print(f'  Total Frame                  : {len(df):,} frame @ {fps_avg:.1f} fps')
print('-' * 62)
print('  [THRESHOLD ADAPTIF]')
print(f'  Threshold Min/Max            : {df["threshold_T_mm"].min():,} / {df["threshold_T_mm"].max():,} mm')
print(f'  Threshold Rata-rata          : {df["threshold_T_mm"].mean():.0f} mm')
print(f'  Kecepatan v_avg Maks         : {df["v_avg_mmps"].max():.0f} mm/s = {df["v_avg_mmps"].max()/1000:.2f} m/s')
print(f'  M_buffer Rata-rata           : {df["m_buffer_mm"].mean():.1f} mm')
print('-' * 62)
print('  [PERINGATAN (ALERT)]')
print(f'  Total Frame Peringatan       : {df["alert_triggered"].sum()} frame')
print(f'  Rasio Frame Peringatan       : {df["alert_triggered"].mean()*100:.1f}%')
print('-' * 62)
print('  [LATENSI END-TO-END]')
print(f'  Latensi Total Mean +/- SD    : {lat_mean:.1f} +/- {lat_std:.1f} ms')
print(f'  Latensi Total Median         : {lat_med:.1f} ms')
print(f'  Latensi Total Maks           : {lat_max} ms')
print(f'  Latensi Jaringan Mean        : {df["latency_net_ms"].mean():.1f} ms')
print(f'  Batas Kognitif t_R           : {T_R*1000:.0f} ms  [Kovacs & Nagy, 2020]')
print(f'  Status                       : {status_lat}')
print('-' * 62)
print('  [RELIABILITAS JARINGAN]')
print(f'  Total Packet Loss            : {total_loss} paket')
print(f'  Packet Delivery Ratio (PDR)  : {pdr:.2f}%')
print(f'  Status (>= 95%?)             : {status_pdr}')
print('=' * 62)

---
## Fig. 4.5 — Confusion Matrix Akurasi Deteksi Arah (Pengujian 3.5.2/2)

**Prasyarat:** CSV harus mengandung baris EVENT (ground truth marker).
Penguji menekan tombol di StreamActivity sebelum tiap trial → `logTestMarker()` menulis:
`EVENT,<timestamp_ms>,<elapsed_s>,GROUND_TRUTH,"Jam 12"`

**Protokol:** 6 kelas × 10 trial = **60 total trial**

| Kelas | Kondisi Aktual |
|:---:|:---|
| Jam 10 | Objek di kolom grid 1-2 (kiri jauh) |
| Jam 11 | Objek di kolom grid 3-4 (kiri depan) |
| Jam 12 | Objek di kolom grid tengah |
| Jam 1  | Objek di kolom grid 5-6 (kanan depan) |
| Jam 2  | Objek di kolom grid 7 (kanan jauh) |
| Jalan Kosong | Tidak ada objek — sistem harus diam |


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Fig. 4.5 — Confusion Matrix Akurasi Deteksi Arah
# ═══════════════════════════════════════════════════════════════════
from io import StringIO
from matplotlib.colors import LinearSegmentedColormap

CLASS_LABELS = ['Jam 10', 'Jam 11', 'Jam 12', 'Jam 1', 'Jam 2', 'Jalan Kosong']
PREDICTION_WINDOW_START_S = -1.0  # mundur 1 detik (toleransi keterlambatan tekan tombol)
PREDICTION_WINDOW_END_S   = 2.0   # maju 2 detik

# ── 1. Pisahkan baris EVENT dari baris data dalam CSV ──────────────────────
def load_csv_with_markers(csv_path):
    raw_lines, event_rows = [], []
    with open(csv_path, 'r', encoding='utf-8') as f:
        hdr = False
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith('EVENT,'):
                p = line.split(',', 5)
                event_rows.append({'event_ts_ms': int(p[1]),
                                   'ground_truth': p[4].strip('"')})
            elif not hdr and 'timestamp_ms' in line:
                raw_lines.append(line); hdr = True
            else:
                raw_lines.append(line)
    df = pd.read_csv(StringIO('\n'.join(raw_lines)))
    return df, pd.DataFrame(event_rows)

# ── 2. Cocokkan setiap marker dengan modus alert_text dalam jendela waktu ──
def _norm(text):
    text = str(text).strip().lower()
    if not text or text in ('nan', '', '0'): return 'Jalan Kosong'
    for k, v in [('10','Jam 10'),('11','Jam 11'),('12','Jam 12'),
                 ('jam 1','Jam 1'),('jam 2','Jam 2'),
                 (' 1','Jam 1'),(' 2','Jam 2'),
                 ('kosong','Jalan Kosong'),('aman','Jalan Kosong')]:
        if k in text: return v
    return 'Jalan Kosong'

def extract_predictions(df_data, df_markers):
    results = []
    for _, m in df_markers.iterrows():
        t0 = m['event_ts_ms'] + PREDICTION_WINDOW_START_S * 1000
        t1 = m['event_ts_ms'] + PREDICTION_WINDOW_END_S * 1000
        win = df_data[(df_data['timestamp_ms'] >= t0) & (df_data['timestamp_ms'] <= t1)]
        if win.empty:
            print(f"[SKIP] Tidak ada frame untuk GT='{m['ground_truth']}'"); continue
        pred = win['alert_text'].apply(_norm).mode().iloc[0]
        results.append({'ground_truth': m['ground_truth'], 'predicted': pred})
    return results

# ── 3. Load CSV & jalankan pipeline ───────────────────────────────────────
df_data, df_markers = load_csv_with_markers(csv_path)
print(f'Frame data: {len(df_data)} | Ground truth markers: {len(df_markers)}')
results = extract_predictions(df_data, df_markers)

# ── 4. Hitung confusion matrix ────────────────────────────────────────────
n_cls    = len(CLASS_LABELS)
lbl_idx  = {l: i for i, l in enumerate(CLASS_LABELS)}
cm = np.zeros((n_cls, n_cls), dtype=int)
for r in results:
    gi, pi = lbl_idx.get(r['ground_truth'], -1), lbl_idx.get(r['predicted'], -1)
    if gi >= 0 and pi >= 0: cm[gi][pi] += 1
accuracy = np.trace(cm) / cm.sum() if cm.sum() > 0 else 0.0
print(f'Akurasi keseluruhan: {accuracy*100:.2f}% (N={cm.sum()})')


In [ ]:
# ── 5. Heatmap + Tabel Metrik per Kelas (Fig. 4.5) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                          gridspec_kw={'width_ratios': [3, 1.5]})

# Heatmap
ax = axes[0]
cmap = LinearSegmentedColormap.from_list('vnetra', ['#FFFFFF','#BDD7EE','#2E75B6'])
im = ax.imshow(cm, cmap=cmap, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n_cls)); ax.set_yticks(range(n_cls))
ax.set_xticklabels(CLASS_LABELS, rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(CLASS_LABELS, fontsize=8)
ax.set_xlabel('Prediksi Sistem', fontsize=9)
ax.set_ylabel('Kondisi Aktual (Ground Truth)', fontsize=9)
ax.set_title(f'Confusion Matrix Deteksi Arah\nAkurasi: {accuracy*100:.1f}%  (N={cm.sum()})', fontsize=10)
for i in range(n_cls):
    for j in range(n_cls):
        clr = 'white' if cm[i,j] > cm.max()*0.6 else 'black'
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=9,
                color=clr, fontweight='bold')

# Tabel metrik
ax2 = axes[1]; ax2.axis('off')
rows = []
for i, lbl in enumerate(CLASS_LABELS):
    tp = cm[i,i]; fp = cm[:,i].sum()-tp; fn = cm[i,:].sum()-tp
    pr = tp/(tp+fp) if (tp+fp)>0 else 0
    rc = tp/(tp+fn) if (tp+fn)>0 else 0
    f1 = 2*pr*rc/(pr+rc) if (pr+rc)>0 else 0
    rows.append([lbl, f'{pr*100:.1f}%', f'{rc*100:.1f}%', f'{f1*100:.1f}%'])
tbl = ax2.table(cellText=rows, colLabels=['Kelas','Precision','Recall','F1'],
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.2, 1.6)
ax2.set_title('Metrik per Kelas', fontsize=10, pad=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_4_5_confusion_matrix.png'),
            dpi=300, bbox_inches='tight')
plt.show()
